In [12]:
import cv2
import numpy as np
import random

In [3]:
image = cv2.imread('image.jpg')
height, width = image.shape[0:2]

# Image Interpolation(resizeing image)

In [16]:
# using _AREA interpolation to resize(Making it smaller) image
area_re_interpol = cv2.resize(image, (int(height/2), int(width/2)), interpolation=cv2.INTER_AREA)

# using LINEAR interpolation to resize(Making it larger) image
linear_re_interpol = cv2.resize(area_re_interpol, None, fx=2, fy = 2, interpolation=cv2.INTER_LINEAR)
# using QUBIC interpolation to resize(Making it larger) image
qubic_re_interpol = cv2.resize(area_re_interpol, None, fx=2, fy = 2, interpolation=cv2.INTER_CUBIC)
# using LANCZOS4 interpolation to resize(Making it larger) image
lancros_re_interpol = cv2.resize(area_re_interpol, None, fx=2, fy = 2, interpolation=cv2.INTER_LANCZOS4)


cv2.imshow('Original', image)
cv2.imshow('Area Interpolation', area_re_interpol)
cv2.imshow('Linear Interpolatio', linear_re_interpol)
cv2.imshow('Qubic Interpolatio', qubic_re_interpol)
cv2.imshow('Lacros Interpolatio', lancros_re_interpol)

# Release the video capture and close any OpenCV windows
if cv2.waitKey(0) & 0xFF == ord('q'):
    cv2.destroyAllWindows()

# Image Translation(Shifting the image)

In [20]:
translation_matrix = np.float32([[1,0,-50],[0,1,75]])

In [27]:
translated = cv2.warpAffine(area_re_interpol, translation_matrix, (area_re_interpol.shape[1]+50, area_re_interpol.shape[0]+75))
cv2.imshow('Area Interpolation', area_re_interpol)

cv2.imshow('Shifted Image', translated)

if cv2.waitKey(0) & 0xFF == ord('q'):
    cv2.destroyAllWindows()

# Image Rotation

In [51]:
rotation_matrix = cv2.getRotationMatrix2D((height/2, width/2), 60, 2)
rotated = cv2.warpAffine(image, rotation_matrix, (width, height))

cv2.imshow('Image', image)

cv2.imshow('Rotated', rotated)

if cv2.waitKey(0) & 0xFF == ord('q'):
    cv2.destroyAllWindows()

error: OpenCV(4.11.0) :-1: error: (-5:Bad argument) in function 'warpAffine'
> Overload resolution failed:
>  - warpAffine() missing required argument 'dsize' (pos 3)
>  - warpAffine() missing required argument 'dsize' (pos 3)


# Image Transformation

## Perspective Transformation(8 possible Transformation)

In [60]:
src_points = np.float32([[0,0], [width-1,0], [0,height-1], [width-1,height-1]])
dest_points = np.float32([[0+10,0],[(width-11), int(0+((height-1)*0.25))],  [0+10,height-1], [width-11,int((height-1)-((height-1)*0.25))]])
perspective_matrix = cv2.getPerspectiveTransform(src_points, dest_points)
z_title_image = cv2.warpPerspective(image, perspective_matrix, (width, height), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
if np.sum(z_title_image) == 0:
    print("The transformed image is empty.")
else:
    print("Transformed image loaded successfully.")
    
# cv2.imshow('Image', image)

cv2.imshow('tilted', z_title_image)

if cv2.waitKey(0) & 0xFF == ord('q'):
    cv2.destroyAllWindows()

Transformed image loaded successfully.


# Color Jitter

In [2]:
def color_jitter(img, brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1):
    img = img.astype(np.float32) / 255.0

    # Brightness
    if brightness > 0:
        alpha_b = 1.0 + random.uniform(-brightness, brightness)
        img = img * alpha_b
        img = np.clip(img, 0, 1)

    # Contrast
    if contrast > 0:
        alpha_c = 1.0 + random.uniform(-contrast, contrast)
        gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2GRAY) / 255.0
        mean = np.mean(gray)
        img = (img - mean) * alpha_c + mean
        img = np.clip(img, 0, 1)

    # Saturation
    if saturation > 0:
        alpha_s = 1.0 + random.uniform(-saturation, saturation)
        hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 1] *= alpha_s
        hsv[..., 1] = np.clip(hsv[..., 1], 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32) / 255.0

    # Hue
    if hue > 0:
        delta_h = random.uniform(-hue, hue) * 180  # Hue range in OpenCV is [0,180]
        hsv = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[..., 0] = (hsv[..., 0] + delta_h) % 180
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR).astype(np.float32) / 255.0

    return (img * 255).astype(np.uint8)

In [14]:
image = cv2.imread('cards1.jpg')
jittered = color_jitter(image)

cv2.imshow("Original", image)
cv2.imshow("Color Jittered", jittered)
cv2.waitKey(0)
cv2.destroyAllWindows()